# Generate PDL Summaries: Pilot
## Can we synthesise a GOI-style `summary` field for PDL companies, and how good is it?

**What this notebook does:**

PDL has no free-text description field (confirmed in notebook 01). This pilot tests whether one can be generated the same way GOI's own `summary` field was built: scrape each company's website, then synthesise a description with an LLM. The pilot is scored against GOI's *real* summary for the same companies, not just eyeballed, by sampling from the 121,300 GOI<->PDL domain overlap, where a genuine ground-truth summary already exists for comparison.

**Pipeline:**
1. Re-stream the PDL file once, keeping full records for companies whose domain also appears in the GOI 300K sample (this reconstructs the overlap set with PDL's own fields, not just a yes/no flag)
2. Sample 400 of those companies
3. Scrape each company's homepage (title, meta description, short visible text, with graceful failure handling)
4. Synthesise a `summary` + `summary_keywords` pair with a local LLM (`qwen2.5:14b` via Ollama), using only PDL's own fields and the scraped text, never GOI's summary
5. Score each generated summary against GOI's real summary for the same company using MiniLM embedding similarity
6. Save everything for manual review

**Runtime note:** scraping 400 sites concurrently (8 workers) should take roughly 10-20 minutes depending on how many sites are slow or unreachable. LLM synthesis for 400 companies on a local model adds roughly another 10-20 minutes depending on GPU.

**Requires:** a local Ollama server running `qwen2.5:14b` (`ollama pull qwen2.5:14b` first if not already present), same `OLLAMA_BASE_URL` convention as `06_hyde_pipeline.ipynb`.

## 1 · Environment Setup

In [4]:
import os, json, re, time, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from bs4 import BeautifulSoup
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

DATASET_DIR = Path('../dataset')
RESULT_DIR = Path('result/02_generate_pdl_summaries_pilot')
RESULT_DIR.mkdir(parents=True, exist_ok=True)

GOI_SAMPLE_PATH = DATASET_DIR / 'goi_random_sample_300k_clean.parquet'
PDL_PATH = DATASET_DIR / 'pdl_companies' / 'free_company_dataset.json'

OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
OLLAMA_MODEL = 'qwen2.5:14b'

PILOT_N = 400
RANDOM_SEED = 42

print(f'[Setup] GOI sample path : {GOI_SAMPLE_PATH} (exists: {GOI_SAMPLE_PATH.exists()})')
print(f'[Setup] PDL path        : {PDL_PATH} (exists: {PDL_PATH.exists()})')
print(f'[Setup] Ollama          : {OLLAMA_BASE_URL}, model {OLLAMA_MODEL}')

[Setup] GOI sample path : ../dataset/goi_random_sample_300k_clean.parquet (exists: True)
[Setup] PDL path        : ../dataset/pdl_companies/free_company_dataset.json (exists: True)
[Setup] Ollama          : http://127.0.0.1:11434, model qwen2.5:14b


## 2 · Rebuild the Overlap Set With Full PDL Records

Same single-pass streaming approach as notebook 01, but this time keeping each matched PDL record in full (not just counting), since the pilot needs PDL's own `industry`/`size`/`founded` fields as LLM input.

In [5]:
def normalise_domain(raw):
    if not raw:
        return None
    d = raw.strip().lower()
    d = re.sub(r'^https?://', '', d)
    d = re.sub(r'^www\.', '', d)
    d = d.split('/')[0]
    d = d.strip()
    return d if d else None

In [ ]:
goi_df = pd.read_parquet(GOI_SAMPLE_PATH)
goi_df['norm_domain'] = goi_df['domain'].apply(normalise_domain)
goi_domains = set(goi_df['norm_domain'].dropna())
print(f'[GOI] Sample rows: {len(goi_df):,}, unique domains: {len(goi_domains):,}')

overlap_pdl_records = {}
t0 = time.time()
with open(PDL_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue
        dom = normalise_domain(rec.get('website'))
        if dom and dom in goi_domains and dom not in overlap_pdl_records:
            overlap_pdl_records[dom] = rec
        if (i + 1) % 10_000_000 == 0:
            print(f'[Stream] {i+1:,} rows processed, {len(overlap_pdl_records):,} matches so far, {time.time()-t0:.0f}s elapsed')

print(f'[Stream] Done. {len(overlap_pdl_records):,} overlap companies with full PDL records, {time.time()-t0:.0f}s elapsed')

## 3 · Sample the Pilot Set

In [ ]:
random.seed(RANDOM_SEED)
pilot_domains = random.sample(sorted(overlap_pdl_records.keys()), PILOT_N)

goi_lookup = goi_df.set_index('norm_domain')
pilot_rows = []
for dom in pilot_domains:
    pdl_rec = overlap_pdl_records[dom]
    goi_rec = goi_lookup.loc[dom]
    pilot_rows.append({
        'domain': dom,
        'pdl_name': pdl_rec.get('name'),
        'pdl_industry': pdl_rec.get('industry'),
        'pdl_size': pdl_rec.get('size'),
        'pdl_country': pdl_rec.get('country'),
        'pdl_founded': pdl_rec.get('founded'),
        'goi_summary_real': goi_rec['summary'],
    })

pilot_df = pd.DataFrame(pilot_rows)
print(f'[Pilot] {len(pilot_df)} companies sampled, seed={RANDOM_SEED}')
pilot_df.head(3)

## 4 · Scrape Each Company's Homepage

Title, meta description, and a short excerpt of visible text, not a full crawl. Concurrent with 8 workers to keep total runtime reasonable. Failures (dead domains, timeouts, blocks) are recorded, not silently dropped.

In [ ]:
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'}
MAX_WORKERS = 8
TIMEOUT_S = 8

def scrape_site(domain):
    for scheme in ('https://', 'http://'):
        try:
            resp = requests.get(f'{scheme}{domain}', headers=HEADERS, timeout=TIMEOUT_S)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, 'html.parser')
            title = soup.title.string.strip() if soup.title and soup.title.string else ''
            meta = soup.find('meta', attrs={'name': 'description'})
            meta_desc = meta['content'].strip() if meta and meta.get('content') else ''
            body_text = ' '.join(soup.get_text(separator=' ', strip=True).split())[:800]
            return {'domain': domain, 'scrape_ok': True, 'title': title, 'meta_description': meta_desc, 'body_excerpt': body_text, 'error': None}
        except Exception as e:
            last_error = str(e)
            continue
    return {'domain': domain, 'scrape_ok': False, 'title': '', 'meta_description': '', 'body_excerpt': '', 'error': last_error}

scrape_results = []
t0 = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(scrape_site, d): d for d in pilot_df['domain']}
    for i, fut in enumerate(as_completed(futures)):
        scrape_results.append(fut.result())
        if (i + 1) % 50 == 0:
            ok = sum(r['scrape_ok'] for r in scrape_results)
            print(f'[Scrape] {i+1}/{len(pilot_df)} done, {ok} succeeded, {time.time()-t0:.0f}s elapsed')

scrape_df = pd.DataFrame(scrape_results)
ok_rate = 100 * scrape_df['scrape_ok'].mean()
print(f'[Scrape] Done. {scrape_df["scrape_ok"].sum()}/{len(scrape_df)} succeeded ({ok_rate:.1f}%)')

pilot_df = pilot_df.merge(scrape_df, on='domain', how='left')

## 5 · Synthesise Summaries With a Local LLM

Uses only PDL's own fields and the scraped text, GOI's real summary is withheld from the prompt entirely, it is only used afterward for scoring.

In [ ]:
PROMPT_TEMPLATE = '''You are writing a short company description for a business search index.

Company name: {name}
Industry label: {industry}
Size: {size}
Country: {country}

Website title: {title}
Website meta description: {meta_description}
Website text excerpt: {body_excerpt}

Write a natural-language company description (2-4 sentences) describing what this company actually does, based only on the information above. If the website text looks like a hosting provider, placeholder page, or unrelated content, say so plainly instead of guessing.

Then list 8-10 short keywords describing the company.

Respond with valid JSON only, in this exact shape:
{{"summary": "...", "summary_keywords": ["...", "..."]}}'''

def synthesise_summary(row):
    prompt = PROMPT_TEMPLATE.format(
        name=row['pdl_name'] or 'unknown',
        industry=row['pdl_industry'] or 'unknown',
        size=row['pdl_size'] or 'unknown',
        country=row['pdl_country'] or 'unknown',
        title=row['title'] or '(none)',
        meta_description=row['meta_description'] or '(none)',
        body_excerpt=row['body_excerpt'] or '(none)',
    )
    try:
        resp = requests.post(
            f'{OLLAMA_BASE_URL}/api/generate',
            json={'model': OLLAMA_MODEL, 'prompt': prompt, 'stream': False, 'format': 'json'},
            timeout=60,
        )
        resp.raise_for_status()
        parsed = json.loads(resp.json()['response'])
        return {'summary_generated': parsed.get('summary', ''), 'keywords_generated': parsed.get('summary_keywords', []), 'llm_ok': True, 'llm_error': None}
    except Exception as e:
        return {'summary_generated': '', 'keywords_generated': [], 'llm_ok': False, 'llm_error': str(e)}

llm_results = []
t0 = time.time()
for i, row in pilot_df.iterrows():
    llm_results.append(synthesise_summary(row))
    if (i + 1) % 50 == 0:
        ok = sum(r['llm_ok'] for r in llm_results)
        print(f'[LLM] {i+1}/{len(pilot_df)} done, {ok} succeeded, {time.time()-t0:.0f}s elapsed')

llm_df = pd.DataFrame(llm_results)
print(f'[LLM] Done. {llm_df["llm_ok"].sum()}/{len(llm_df)} succeeded, {time.time()-t0:.0f}s total')

pilot_df = pd.concat([pilot_df.reset_index(drop=True), llm_df], axis=1)

## 6 · Score Against GOI's Real Summary

Embedding cosine similarity via MiniLM (the same model used throughout this thesis), between the generated summary and GOI's actual summary for the same company.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

scored_mask = pilot_df['llm_ok'] & pilot_df['summary_generated'].str.len().gt(0)
scored_df = pilot_df[scored_mask].copy()

emb_generated = model.encode(scored_df['summary_generated'].tolist(), show_progress_bar=False)
emb_real = model.encode(scored_df['goi_summary_real'].tolist(), show_progress_bar=False)

cos_sim = np.sum(emb_generated * emb_real, axis=1) / (np.linalg.norm(emb_generated, axis=1) * np.linalg.norm(emb_real, axis=1))
scored_df['similarity_to_real_goi_summary'] = cos_sim

pilot_df = pilot_df.merge(scored_df[['domain', 'similarity_to_real_goi_summary']], on='domain', how='left')

print(f'[Score] Scored {len(scored_df)} of {len(pilot_df)} pilot companies')
print(f'[Score] Mean similarity   : {cos_sim.mean():.3f}')
print(f'[Score] Median similarity : {np.median(cos_sim):.3f}')
print(f'[Score] Below 0.3 (likely poor match): {(cos_sim < 0.3).sum()} ({100*(cos_sim < 0.3).mean():.1f}%)')
print(f'[Score] Above 0.6 (likely good match) : {(cos_sim > 0.6).sum()} ({100*(cos_sim > 0.6).mean():.1f}%)')

## 7 · Spot-Check: A Few Side-by-Side Examples

In [ ]:
sample_check = scored_df.sample(min(5, len(scored_df)), random_state=RANDOM_SEED)
for _, row in sample_check.iterrows():
    print(f'--- {row["domain"]} (similarity: {row["similarity_to_real_goi_summary"]:.3f}) ---')
    print(f'[Real GOI]      {row["goi_summary_real"][:300]}')
    print(f'[Generated PDL] {row["summary_generated"][:300]}')
    print()

## 8 · Save Everything

In [ ]:
pilot_df.to_csv(RESULT_DIR / 'pilot_results.csv', index=False)

summary_stats = {
    'pilot_n': len(pilot_df),
    'scrape_success_rate_pct': round(100 * pilot_df['scrape_ok'].mean(), 1),
    'llm_success_rate_pct': round(100 * pilot_df['llm_ok'].mean(), 1),
    'mean_similarity_to_real_goi_summary': round(float(cos_sim.mean()), 3),
    'median_similarity_to_real_goi_summary': round(float(np.median(cos_sim)), 3),
    'pct_below_0.3_similarity': round(100 * float((cos_sim < 0.3).mean()), 1),
    'pct_above_0.6_similarity': round(100 * float((cos_sim > 0.6).mean()), 1),
    'ollama_model': OLLAMA_MODEL,
}

with open(RESULT_DIR / 'pilot_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary_stats, f, indent=2)

print(f'[Save] Written to {RESULT_DIR}/')
print(json.dumps(summary_stats, indent=2))